# Bootstrap Confidence Intervals for Synthetic Results

Synthetic-data equivalent of `bootstrapping-bulk_apply_confidence_intervals_to_results.ipynb`.

For every configuration in the synthetic results, this notebook:
1. Loads the segment-level predictions from the parquet file.
2. Computes the CCC with 95% participant-clustered bootstrap confidence intervals.
3. Writes the enriched results back as `-conf.csv` files.

**Input:** `results/synthetic/composed/everything/results-{target}.csv`  
**Output:** `results/synthetic/composed/everything/results-{target}-conf.csv`

In [1]:
import os
import pandas as pd
import audbenchmark
from tqdm import tqdm

# Import bootstrapping
from confidence_intervals import evaluate_with_conf_int

In [2]:
path_composed = "../../results/synthetic/composed"
path_results_base = os.path.abspath(os.path.join(path_composed, "..", "modelling"))
metric = audbenchmark.metric.concordance_cc

# The synthetic collect_results.py writes to the "everything" sub-directory
lst_conditions = ["everything"]

In [3]:
meta_data = []

for cur_condition in tqdm(lst_conditions, desc="Conditions"):
    path_condition = os.path.join(path_composed, cur_condition)

    csv_files = [
        f
        for f in os.listdir(path_condition)
        if f.endswith(".csv") and not f.endswith("-conf.csv")
    ]

    for file_name in tqdm(csv_files, desc=f"Targets {cur_condition}", leave=False):
        file_path = os.path.join(path_condition, file_name)
        df_results = pd.read_csv(file_path)

        target = file_name.replace("results-", "").replace(".csv", "")

        ccc_conf_mean = []
        ccc_conf_low = []
        ccc_conf_high = []
        lower_bound_larger_null = []
        tasks = []
        features = []

        for str_path in tqdm(
            df_results["path"],
            desc=f"bootstrapping {target}",
            leave=False,
        ):
            str_path_clean = str_path.lstrip("/")

            # Extract task and features from the path components
            path_parts = str_path_clean.split("/")
            task = path_parts[2] if len(path_parts) > 2 else ""
            feature = path_parts[7] if len(path_parts) > 7 else ""

            str_path_data = str_path_clean.split("models/results-compiled.yaml")[0]
            path = os.path.join(
                path_results_base, str_path_data, "data/df_results_test.parquet.zstd"
            )

            df_test = pd.read_parquet(path)

            tpl_result = evaluate_with_conf_int(
                samples=df_test["predictions"].values,
                metric=metric,
                labels=df_test[target].values,
                conditions=df_test["participant_code"].values,
                num_bootstraps=1000,
                alpha=5,
            )

            mean_value = tpl_result[0]
            conf_int_low = tpl_result[1][0]
            conf_int_high = tpl_result[1][1]

            ccc_conf_mean.append(mean_value)
            ccc_conf_low.append(conf_int_low)
            ccc_conf_high.append(conf_int_high)
            lower_bound_larger_null.append(conf_int_low > 0)
            tasks.append(task)
            features.append(feature)

        df_results.insert(0, "ccc_conf_mean", ccc_conf_mean)
        df_results.insert(1, "ccc_conf_low", ccc_conf_low)
        df_results.insert(2, "ccc_conf_high", ccc_conf_high)
        df_results.insert(3, "lower_bound_larger_null", lower_bound_larger_null)
        df_results.insert(4, "task", tasks)
        df_results.insert(5, "features", features)

        df_results = df_results.sort_values(by="ccc_conf_mean", ascending=False)

        new_file_path = file_path.replace(".csv", "-conf.csv")
        df_results.to_csv(new_file_path, index=False)
        print(f"Wrote {new_file_path}")

Conditions:   0%|          | 0/1 [00:00<?, ?it/s]







Wrote ../../results/synthetic/composed/everything/results-who_5_percentage_score_corrected-conf.csv


Wrote ../../results/synthetic/composed/everything/results-pss_10_total_score-conf.csv


Wrote ../../results/synthetic/composed/everything/results-stress_current-conf.csv


Wrote ../../results/synthetic/composed/everything/results-phq_8_total_score-conf.csv








Conditions: 100%|██████████| 1/1 [01:55<00:00, 115.22s/it]

Wrote ../../results/synthetic/composed/everything/results-stress_work_tasks-conf.csv
